# VeritasAI · Approach B · Frozen backbone + temporal head
Backbone runs ONCE per frame (features cached), then tiny temporal heads train in seconds.
This is the cheap, upgraded version of the repo's original ResNeXt50+LSTM. GPU only for the feature step (~5-10 min).

In [ ]:
# ==== 0. SETUP (same in every notebook) ===================================
# Kaggle: Settings -> Accelerator = GPU T4 x2 (or None for notebook 00), Internet = ON
# Colab : Runtime -> Change runtime type -> T4 GPU
import os, sys
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO_URL = "https://github.com/Bhoomified/VeritasAI.git"   # your repo (push the new ml/ files first!)
os.chdir(WORK)
if not os.path.exists("VeritasAI"):
    os.system(f"git clone -q {REPO_URL}")
os.chdir(f"{WORK}/VeritasAI"); os.system("git pull -q")
os.system('pip -q install "opencv-python-headless==4.10.0.84" onnx onnxruntime tabulate')
if not KAGGLE:   # Colab: keep outputs on Google Drive so a disconnect never loses work
    from google.colab import drive; drive.mount("/content/drive")
STORE = "/kaggle/working" if KAGGLE else "/content/drive/MyDrive/veritas"
os.makedirs(STORE, exist_ok=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ==== where the face strips from notebook 00 live ==========================
# Kaggle: "Add Input" -> your notebook-00 output; it appears under /kaggle/input/<name>/
FACES = "/kaggle/input/veritas-faces/faces" if KAGGLE else f"{STORE}/faces"
TRAIN_SET, CROSS_SET = "ffpp", "celebdf"          # train on FF++, test unseen on Celeb-DF
F = f"{FACES}/{TRAIN_SET}"
TR, VA, TE = f"{F}/{TRAIN_SET}_train.csv", f"{F}/{TRAIN_SET}_val.csv", f"{F}/{TRAIN_SET}_test.csv"
CROSS = f"{FACES}/{CROSS_SET}/{CROSS_SET}_test.csv"
RESULTS = f"{STORE}/results.csv"; CK = f"{STORE}/checkpoints"

In [ ]:
# ==== cache features for 3 backbones (light -> heavy) ======================
BACKBONES = ["mobilenet_v3_large", "efficientnet_b0", "resnext50_32x4d"]   # resnext = original repo baseline
for bb in BACKBONES:
    !python -m ml.features --faces {F} --index {TR} {VA} {TE} --backbone {bb} --out {STORE}/feats/{TRAIN_SET}_{bb} --bs 16 --workers 4
    !python -m ml.features --faces {FACES}/{CROSS_SET} --index {CROSS} --backbone {bb} --out {STORE}/feats/{CROSS_SET}_{bb} --bs 16 --workers 4

In [ ]:
# ==== grid: every backbone x every temporal head, 3 seeds each ==============
for bb in BACKBONES:
    for head in ["mean", "lstm", "gru", "tcn", "attn"]:
        !python -m ml.train_head --feats {STORE}/feats/{TRAIN_SET}_{bb} --head {head} --seeds 3 \
            --cross {STORE}/feats/{CROSS_SET}_{bb}/test.npz --out {CK} --results {RESULTS} | tail -2
import pandas as pd; pd.read_csv(RESULTS)

Tip: after the feature step you can download `feats/` (a few hundred MB) and run the head grid on a laptop CPU in VS Code.